In [3]:
import pandas as pd
import tqdm

from src.electricity_price_forecasting.data.data_pipeline import fetch_data

merged_df, quarterly_df, hourly_df = fetch_data(pd.Timestamp("2024-07-01"), pd.Timestamp("2025-04-01"))

2025-04-02 16:40:00,816 - INFO - Fetching 'CZ' data from ENTSO-E...
2025-04-02 16:40:01,225 - INFO - Fetching data for CZ between 2025-03-07 00:00:00+01:00 and 2025-04-06 01:00:00+02:00
2025-04-02 16:40:03,350 - INFO - Fetching data for CZ between 2025-03-06 00:00:00+01:00 and 2025-04-05 01:00:00+02:00
2025-04-02 16:40:07,631 - INFO - Fetching data for CZ between 2025-03-07 00:00:00+01:00 and 2025-04-06 01:00:00+02:00
2025-04-02 16:40:09,870 - INFO - Fetching data from OTE CR...
2025-04-02 16:40:10,076 - INFO - Fetching OTE data from https://www.ote-cr.cz/en/short-term-markets/electricity/day-ahead-market/@@chart-data?report_date=2025-03-06
2025-04-02 16:40:10,310 - INFO - Fetching OTE data from https://www.ote-cr.cz/en/short-term-markets/electricity/day-ahead-market/@@chart-data?report_date=2025-03-07
2025-04-02 16:40:10,546 - INFO - Fetching OTE data from https://www.ote-cr.cz/en/short-term-markets/electricity/day-ahead-market/@@chart-data?report_date=2025-03-08
2025-04-02 16:40:10,7

In [49]:
merged_df.ffill(inplace=True)

In [16]:
merged_df.describe()

,Forecasted Load,Actual Aggregated,Actual Consumption,Solar,DA Volume (MWh),DA Price (EUR/MWh),ID Volume (MWh),ID Weighted average of the prices (EUR/MWh),ID Minimal price (EUR/MWh),ID Maximal price (EUR/MWh),...,55_22_wind_gusts_10m,55_22_direct_radiation,55_23_temperature_2m,55_23_rain,55_23_wind_speed_10m,55_23_wind_speed_80m,55_23_wind_direction_10m,55_23_wind_direction_80m,55_23_wind_gusts_10m,55_23_direct_radiation
count,26304.000000,26301.000000,26301.0,26301.000000,6576.000000,6576.000000,26304.000000,26264.000000,26264.000000,26264.000000,...,26300.000000,26300.000000,26300.000000,26300.000000,26300.000000,26300.000000,26300.000000,26300.000000,26300.000000,26300.000000
mean,7185.270871,7916.280560,0.0,411.532793,3406.202661,105.924112,155.523080,105.055987,43.537854,187.588993,...,24.078647,43.377977,8.571086,0.015091,10.853069,21.089051,191.690913,197.937581,24.039390,42.440433
std,1378.581394,1865.987048,0.0,680.121197,780.309460,56.803430,85.106648,63.397922,86.216538,245.660573,...,13.421934,102.851307,8.727746,0.075027,6.212328,10.012644,88.579840,87.279462,12.898095,100.188820
min,3935.000000,5.000000,0.0,0.000000,1525.100000,-81.160000,0.000000,-509.940000,-4977.000000,-40.000000,...,0.000000,-1.726299,-9.550000,0.000000,0.360000,0.360000,0.000000,0.578713,1.800000,-1.729008
25%,6185.000000,6947.000000,0.0,0.000000,2860.100000,80.240000,93.600000,76.500000,10.000000,119.452500,...,14.400000,0.000000,1.500000,0.000000,6.120000,13.684735,131.000000,136.755687,14.400000,0.000000
50%,7171.500000,8025.000000,0.0,2.000000,3364.850000,104.950000,141.900000,101.000000,49.050000,152.830000,...,21.960000,0.000000,5.750000,0.000000,9.720000,20.898611,203.000000,210.487815,21.960000,0.000000
75%,8154.000000,9090.000000,0.0,572.000000,3912.125000,130.480000,200.900000,128.952500,82.000000,199.990000,...,31.319998,12.725759,15.900000,0.000000,14.759999,27.553525,261.000000,267.110518,31.680000,11.730459
max,10853.000000,13104.000000,0.0,3065.000000,6084.800000,844.630000,818.500000,1393.770000,500.000000,9998.990000,...,100.799995,612.859600,33.600000,2.400000,41.399998,72.667410,360.000000,360.000000,91.799995,622.703370


In [17]:
base_weather_features = [
    "temperature_2m",
    "rain",
    "wind_speed_10m",
    "wind_speed_80m",
    "wind_direction_10m",
    "wind_direction_80m",
    "wind_gusts_10m",
    "direct_radiation"
]

weather_features = [col for col in merged_df.columns if any([feature in col for feature in base_weather_features])]
intraday_features = ["ID Weighted average of the prices (EUR/MWh)"]
day_ahead_features = ["DA Price (EUR/MWh)"]

In [18]:
num_day_ahead_features = len(day_ahead_features)
num_intraday_features = len(intraday_features)
num_weather_features = len(weather_features)

past_timesteps = 7 * 24 * 4
num_past_features = num_day_ahead_features + num_weather_features

today_timesteps = 24 * 4
num_today_features = num_weather_features
num_target_features = num_day_ahead_features

In [23]:
import torch

class WeatherPriceDataset(torch.utils.data.Dataset):
    def __init__(self, df, past_timesteps, today_timesteps, weather_features, day_ahead_features):
        self.df = df
        self.past_timesteps = past_timesteps
        self.today_timesteps = today_timesteps
        self.weather_features = weather_features
        self.day_ahead_features = day_ahead_features

    def __len__(self):
        return len(self.df) - self.past_timesteps - self.today_timesteps - 1

    def __getitem__(self, idx):
        past_weather = self.df.iloc[idx:idx+self.past_timesteps][self.weather_features].values
        past_price = self.df.iloc[idx:idx+self.past_timesteps][self.day_ahead_features].values
        today_weather = self.df.iloc[idx+self.past_timesteps:idx+self.past_timesteps+self.today_timesteps][self.weather_features].values
        today_price = self.df.iloc[idx+self.past_timesteps:idx+self.past_timesteps+self.today_timesteps][self.day_ahead_features].values

        return {
            'past_weather': past_weather,
            'past_price': past_price,
            'today_weather': today_weather,
            'today_price': today_price,
        }

In [40]:
class DayAheadPriceRNN(torch.nn.Module):
    def __init__(self, past_input_size, today_input_size, hidden_size, output_size):
        super(DayAheadPriceRNN, self).__init__()
        self.encoder_rnn = torch.nn.GRU(input_size=past_input_size, hidden_size=hidden_size, batch_first=True)
        self.decoder_rnn = torch.nn.GRU(input_size=today_input_size, hidden_size=hidden_size, batch_first=True)
        self.fc_out = torch.nn.Linear(hidden_size, output_size)

    def forward(self, past_weather, past_price, today_weather):
        past_input = torch.cat([past_weather, past_price], dim=-1)  # (B, T_past, F1+F2)
        _, hidden = self.encoder_rnn(past_input)  # hidden: (1, B, H)
        decoder_output, _ = self.decoder_rnn(today_weather, hidden)  # (B, T_today, H)
        price_pred = self.fc_out(decoder_output)  # (B, T_today, output_size)
        return price_pred

In [55]:
from tqdm import tqdm

def train(model, train_loader, val_loader, criterion, optimizer, device, epochs=10):
    model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            past_weather = batch['past_weather'].float().to(device)
            past_price = batch['past_price'].float().to(device)
            today_weather = batch['today_weather'].float().to(device)
            today_price = batch['today_price'].float().to(device)

            output = model(past_weather, past_price, today_weather)
            loss = criterion(output, today_price)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                past_weather = batch['past_weather'].float().to(device)
                past_price = batch['past_price'].float().to(device)
                today_weather = batch['today_weather'].float().to(device)
                today_price = batch['today_price'].float().to(device)

                output = model(past_weather, past_price, today_weather)
                loss = criterion(output, today_price)
                val_loss += loss.item()

        print(f"Epoch {epoch+1}: Train Loss = {train_loss/len(train_loader):.4f}, "
              f"Val Loss = {val_loss/len(val_loader):.4f}")


In [57]:
from sklearn.model_selection import train_test_split

day_ahead_dataset = WeatherPriceDataset(merged_df, past_timesteps, today_timesteps, weather_features, day_ahead_features)

indices = list(range(len(day_ahead_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, shuffle=True, random_state=42)

In [58]:
from torch.utils.data import DataLoader

train_dataset = WeatherPriceDataset(merged_df.iloc[train_idx], past_timesteps, today_timesteps, weather_features, day_ahead_features)
val_dataset = WeatherPriceDataset(merged_df.iloc[val_idx], past_timesteps, today_timesteps, weather_features, day_ahead_features)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
model = DayAheadPriceRNN(num_weather_features + num_day_ahead_features, num_weather_features, 64, num_day_ahead_features)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train(model, train_loader, val_loader, criterion, optimizer, device, epochs=20)

Epoch 1/20 [Train]:  13%|█▎        | 83/615 [00:32<03:08,  2.82it/s]